In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
# Load the FAQ documents and the search index:

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
# Create a lookup table for the original FAQ documents:

doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [4]:

import os 
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
llm_model = "llama-3.1-8b-instant"

In [5]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    model = llm_model
)

In [6]:
# Run RAG for one question:

rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

"Yes, it is okay to join the course late if you just found it now. However, keep in mind the following:\n\n1. If you want to receive a certificate, you need to submit your project while the submissions are still open. Check the course deadline for submissions to ensure you have enough time to complete the project and submit it on time.\n\n2. Since the capstone project is an individual project, there's no issue with joining late, as long as you are aware of the above point regarding certificate eligibility.\n\n3. Be aware of the environment setup options, and if you decide to run the course locally, make sure to document your setup and keep your environment reproducible.\n\nIn terms of technical setup, the course provides flexibility in choosing between running it on Codespaces or locally. Both options are available for those who are comfortable with the necessary tools. \n\nOverall, it seems that the course administrators are open to late joiners and encourage students to start learnin

In [7]:
# Check the cost of this call:

assistant.total_cost()

0.00124425

In [8]:
# Get the original answer from the document ID:

doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [9]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

In [10]:
# Create a function that processes one ground truth record:

def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [11]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Based on the provided information, it appears that you can still join the course late, and there are no restrictions on registering or submitting assignments. However, if you want to receive a certificate, you need to submit your project while submissions are still open. \n\nThere are also specific details related to the format of the capstone project, such as it being an individual project even though collaboration is allowed. Additionally, the course environment is modular, and you can run the course locally using the same setup and tools provided by Codespaces.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [12]:
assistant.reset_usage()

In [13]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [14]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/395 [00:00<?, ?it/s]

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01ks7h6m1ffk3tt9zb0yx6496w` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5611, Requested 1016. Please try again in 6.27s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [ ]:
assistant.total_cost()

In [15]:
!cp /home/hsu/Documents/Learning/llm-zoomcamp/04-evaluation/data/rag-answers-new.csv ./data/rag-answers-new.csv

In [ ]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)